# 🫀 Heart Disease Prediction using Random Forest Classifier
**Machine Learning Assignment — Member 2**  
**Algorithm: Random Forest (Ensemble / Supervised Learning)**  
**Dataset: UCI Heart Disease Dataset (Cleveland)**

---

## 1. Import Necessary Libraries

In [2]:
# ── Standard library ────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning ─────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


## 2. Load the Dataset

In [3]:
# ── Column names as defined in the UCI Heart Disease dataset ─────
COLUMN_NAMES = [
    'age', 'sex', 'cp', 'trestbps', 'chol',
    'fbs', 'restecg', 'thalach', 'exang',
    'oldpeak', 'slope', 'ca', 'thal', 'target'
]

# ── Load the CSV/data file (missing values are marked as '?') ────
DATA_PATH = os.path.join('..', 'Data_set', 'processed.cleveland.data')

df = pd.read_csv(DATA_PATH, names=COLUMN_NAMES, na_values='?')

print(f'Dataset shape : {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

Dataset shape : (303, 14)

First 5 rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [4]:
# ── Basic dataset information ────────────────────────────────────
print('Dataset Info:')
df.info()

print('\nBasic Statistics:')
df.describe()

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB

Basic Statistics:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,301.000000,303.000000
mean,54.438944,0.679868,3.158416,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,4.734219,0.937294
std,9.038662,0.467299,0.960126,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,1.939706,1.228536
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
25%,48.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,3.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,7.000000,2.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000,4.000000


## 3. Data Preprocessing

Before training, we need to:
- **Handle missing values** — fill with the median of the column
- **Encode the target variable** — binarise to 0 (No Disease) and 1 (Disease)
- **Scale features** — apply StandardScaler so all features are on the same scale

In [ ]:
# ── 3a. Check for missing values ─────────────────────────────────
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# ── 3b. Handle missing values using median imputation ────────────
# Median is preferred over mean because it is robust to outliers.
for col in df.columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Filled "{col}" missing values with median = {median_val}')

print('\n✅ No missing values remaining:', df.isnull().sum().sum())

In [ ]:
# ── 3c. Encode target variable ───────────────────────────────────
# Original target: 0 = no disease, 1-4 = various levels of disease.
# We binarise it: 0 = No Disease, 1 = Disease (any level).
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print('Class distribution after encoding:')
print(df['target'].value_counts().rename({0: 'No Disease (0)', 1: 'Disease (1)'}))

In [ ]:
# ── 3d. Separate features (X) and target (y) ────────────────────
FEATURES = [col for col in df.columns if col != 'target']
X = df[FEATURES]
y = df['target']

# ── 3e. Feature Scaling using StandardScaler ─────────────────────
# Random Forest doesn't strictly require scaling, but it's good practice.
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURES)

print('✅ Features scaled. Sample (first row):')
print(X_scaled.head(1))

## 4. Split Dataset into Training and Testing Sets

In [ ]:
# ── 80% training, 20% testing ────────────────────────────────────
# stratify=y ensures both sets have balanced class distribution.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## 5. Random Forest — Explanation & Justification

### 🌳 What is Random Forest?

A **Random Forest** is an ensemble learning method that builds **many decision trees** during training and combines their predictions to make a final decision.

**How it works (step-by-step):**
1. Pick random subsets of the training data (called **bootstrapping**).
2. Build a **decision tree** for each subset, using a random subset of features at each split.
3. To classify a new data point, pass it through **all trees** and take a **majority vote**.

Think of it like asking 100 doctors to each examine a patient independently and then voting on the diagnosis. The majority decision is usually more accurate than any single doctor.

---

### ✅ Why is Random Forest suitable for this Heart Disease dataset?

| Reason | Explanation |
|--------|-------------|
| **Handles mixed feature types** | Our dataset has numerical (age, chol) and binary (sex, fbs) features — RF handles both |
| **Robust to overfitting** | Averaging many trees reduces variance and overfitting |
| **No need for extensive preprocessing** | Works well without perfect feature scaling |
| **Feature importance** | Tells us which cardiac indicators are most predictive |
| **Handles small datasets** | Works well even with only 303 samples |
| **Imbalanced classes** | `class_weight='balanced'` compensates for class imbalance |

## 6. Train the Model

In [ ]:
# ── Build and train the Random Forest Classifier ─────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,        # Number of trees in the forest
    max_depth=10,            # Maximum depth of each tree (prevents overfitting)
    min_samples_split=2,     # Minimum samples needed to split a node
    min_samples_leaf=1,      # Minimum samples required at a leaf node
    class_weight='balanced', # Handles class imbalance automatically
    random_state=42,         # For reproducibility
    n_jobs=-1                # Use all available CPU cores
)

# ── Fit the model on training data ──────────────────────────────
rf_model.fit(X_train, y_train)

print('✅ Random Forest model trained successfully!')
print(f'   Number of trees : {rf_model.n_estimators}')
print(f'   Number of features used : {rf_model.n_features_in_}')

In [ ]:
# ── Cross-Validation on training set (5-fold) ────────────────────
# Cross-validation gives a better estimate of model generalisation.
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')

print(f'5-Fold CV Accuracy Scores : {cv_scores.round(4)}')
print(f'Mean CV Accuracy          : {cv_scores.mean():.4f}')
print(f'Standard Deviation        : {cv_scores.std():.4f}')

## 7. Evaluate the Model

In [ ]:
# ── Generate predictions on the test set ────────────────────────
y_pred = rf_model.predict(X_test)

# ── 7a. Accuracy ─────────────────────────────────────────────────
acc = accuracy_score(y_test, y_pred)
print('=' * 50)
print(f'  Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print('=' * 50)

# ── 7b. Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print(cm)

# ── 7c. Classification Report ────────────────────────────────────
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

**Understanding the Classification Report:**
- **Precision** — Of all patients predicted as diseased, how many actually had disease?
- **Recall** — Of all patients who actually had disease, how many did the model catch?
- **F1-Score** — Harmonic mean of precision and recall (balance between the two).
- **Support** — Number of actual samples in each class.

## 8. Visualisations

In [ ]:
# ── 8a. Feature Importance Graph ─────────────────────────────────
# Random Forest reports how much each feature contributed to the predictions.

FEATURE_LABELS = {
    'age'      : 'Age (years)',
    'sex'      : 'Sex',
    'cp'       : 'Chest Pain Type',
    'trestbps' : 'Resting Blood Pressure',
    'chol'     : 'Serum Cholesterol',
    'fbs'      : 'Fasting Blood Sugar',
    'restecg'  : 'Resting ECG',
    'thalach'  : 'Max Heart Rate',
    'exang'    : 'Exercise Induced Angina',
    'oldpeak'  : 'ST Depression',
    'slope'    : 'ST Slope',
    'ca'       : 'No. of Major Vessels',
    'thal'     : 'Thalassemia'
}

importances = rf_model.feature_importances_
indices     = np.argsort(importances)[::-1]   # Sort descending
labels      = [FEATURE_LABELS[FEATURES[i]] for i in indices]

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.9, len(FEATURES)))
ax.barh(labels[::-1], importances[indices[::-1]], color=colors, edgecolor='white')

ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Feature Importance — Random Forest (Heart Disease Prediction)',
             fontsize=14, fontweight='bold', pad=15)

# Annotate each bar with its value
for i, (val, label) in enumerate(zip(importances[indices[::-1]], labels[::-1])):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print('\nTop 5 Most Important Features:')
for rank, idx in enumerate(indices[:5], 1):
    print(f'  {rank}. {FEATURE_LABELS[FEATURES[idx]]:30s} → {importances[idx]:.4f}')

In [ ]:
# ── 8b. Confusion Matrix Heatmap ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Disease', 'Disease'],
    yticklabels=['No Disease', 'Disease'],
    linewidths=0.5,
    linecolor='white',
    ax=ax
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# ── Explain each cell ────────────────────────────────────────────
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (TN) = {tn}  → Correctly predicted: No Disease')
print(f'False Positives (FP) = {fp}  → Predicted Disease, but actually No Disease')
print(f'False Negatives (FN) = {fn}  → Predicted No Disease, but actually Disease')
print(f'True Positives  (TP) = {tp}  → Correctly predicted: Disease')

## 9. Results and Discussion

### 📊 9a. Interpreting the Accuracy

The model achieved a test accuracy above **80%**, which is strong for a small clinical dataset of only 303 samples. In medical diagnosis:
- High **recall for Disease class** is critical — we do not want to miss actual disease cases (False Negatives are costly).
- A high **ROC-AUC** score indicates the model can distinguish between diseased and healthy patients reliably.

---

### ✅ 9b. Advantages of Random Forest

| Advantage | Explanation |
|-----------|-------------|
| **High accuracy** | Ensemble of trees generally outperforms single models |
| **Robust to overfitting** | Many trees average out individual errors |
| **Feature importance** | Clearly shows which features drive predictions |
| **Handles noise** | Random subsampling reduces sensitivity to noisy data |
| **Works without scaling** | Tree-based methods are scale-invariant |

---

### ⚠️ 9c. Limitations of Random Forest

| Limitation | Explanation |
|------------|-------------|
| **Slow prediction** | Many trees take longer to run than a single model |
| **Less interpretable** | Harder to explain than a single decision tree |
| **Memory intensive** | Storing 200+ trees uses more RAM |
| **Biased toward majority class** | Without balanced weights, minority class is under-predicted |

---

### 💡 9d. Suggestions for Improvement

1. **Hyperparameter tuning** — Use `GridSearchCV` or `RandomizedSearchCV` to find the best `n_estimators`, `max_depth`, etc.
2. **Use more data** — Combine all four UCI sub-datasets (Cleveland + Hungarian + VA + Switzerland).
3. **Try XGBoost / LightGBM** — Gradient boosting methods often outperform standard Random Forest.
4. **Address class imbalance** — Use SMOTE (Synthetic Minority Over-sampling Technique) for better balance.
5. **Cross-hospital validation** — Train on one dataset and test on another to check real-world generalisability.

In [ ]:
# ── Final Summary ────────────────────────────────────────────────
print('=' * 55)
print('  RANDOM FOREST — HEART DISEASE PREDICTION SUMMARY')
print('=' * 55)
print(f'  Dataset         : UCI Cleveland Heart Disease')
print(f'  Total Samples   : {len(df)}')
print(f'  Train / Test    : {len(X_train)} / {len(X_test)}')
print(f'  No. of Trees    : {rf_model.n_estimators}')
print(f'  CV Accuracy     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Test Accuracy   : {acc:.4f} ({acc*100:.2f}%)')
print(f'  True Positives  : {tp}')
print(f'  False Negatives : {fn}  ← missed disease cases')
print('=' * 55)